# 🎙️ OmniVoice 보이스 클로닝 (Colab A100)

[k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice)로 `voice_01.wav` 목소리를 복제해, 텍스트를 넣으면 음성 파일을 만듭니다.

**사용법**
1. 메뉴 **런타임 → 런타임 유형 변경 → A100 GPU** 선택
2. 1~4번 셀을 위에서부터 한 번씩 실행 (3번 셀에서 `voice_01.wav` 업로드)
3. **5번 셀**에 텍스트를 적고 실행 → 음성 재생 + wav 다운로드. 텍스트만 바꿔서 5번만 반복 실행하면 됩니다.

In [ ]:
#@title 1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. OmniVoice 설치 (2~3분)
!pip install -q omnivoice
# import 오류(torch 버전 문제)가 나면 아래 줄 주석을 풀고 다시 실행한 뒤, 런타임 > 세션 다시 시작
# !pip install -q torch==2.8.0+cu128 torchaudio==2.8.0+cu128 --extra-index-url https://download.pytorch.org/whl/cu128
import omnivoice, torch
print("omnivoice OK | torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0))

In [ ]:
#@title 3. 레퍼런스 음성(voice_01.wav) 업로드
#@markdown 이미 `/content/voice_01.wav`가 있으면 업로드를 건너뜁니다. 3~10초, 잡음 없는 음성이 좋습니다.
import os
from google.colab import files
from IPython.display import Audio, display

REF_AUDIO = "/content/voice_01.wav"  #@param {type:"string"}
if not os.path.exists(REF_AUDIO):
    up = files.upload()  # voice_01.wav 선택
    name = next(iter(up))
    os.rename(name, REF_AUDIO) if os.path.abspath(name) != REF_AUDIO else None
print("레퍼런스:", REF_AUDIO)
display(Audio(REF_AUDIO))

In [ ]:
#@title 4. 모델 로드 + 목소리 등록 (최초 1회)
#@markdown `REF_TEXT`: 레퍼런스 음성에서 실제로 말한 문장. **비워두면 Whisper가 자동 받아쓰기**합니다(정확히 적으면 품질↑).
REF_TEXT = ""  #@param {type:"string"}

import torch
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=not REF_TEXT.strip(),  # ref_text가 없을 때만 Whisper 로드
)

voice_prompt = model.create_voice_clone_prompt(
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT.strip() or None,
)
voice_prompt.save("/content/my_voice.pt")  # 다음 세션에서 VoiceClonePrompt.load()로 재사용 가능
print("✅ 목소리 등록 완료")

In [ ]:
#@title 5. ✍️ 텍스트 → 음성 생성 (텍스트만 바꿔서 반복 실행)
TEXT = "안녕하세요. OmniVoice로 만든 제 목소리입니다. 오늘도 좋은 하루 보내세요."  #@param {type:"string"}
SPEED = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.05}
NUM_STEP = 32  #@param [16, 32] {type:"raw"}
FILE_NAME = ""  #@param {type:"string"}
AUTO_DOWNLOAD = True  #@param {type:"boolean"}

import os, time
import soundfile as sf
from google.colab import files
from IPython.display import Audio, HTML, display
import base64

def show_with_download(path, mime="audio/wav"):
    # 플레이어 + 다운로드 버튼 표시
    if mime.startswith("audio"):
        display(Audio(path))
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    fn = os.path.basename(path)
    display(HTML(f'<a download="{fn}" href="data:{mime};base64,{b64}" '
                 f'style="display:inline-block;padding:8px 16px;background:#1a73e8;color:#fff;'
                 f'border-radius:6px;text-decoration:none;font-weight:bold">⬇ 다운로드: {fn}</a>'))

os.makedirs("/content/outputs", exist_ok=True)
t0 = time.time()
audio = model.generate(
    text=TEXT,
    voice_clone_prompt=voice_prompt,
    speed=SPEED,
    num_step=NUM_STEP,
)
name = (FILE_NAME.strip() or time.strftime("out_%Y%m%d_%H%M%S"))
out_path = f"/content/outputs/{name.removesuffix('.wav')}.wav"
sf.write(out_path, audio[0], 24000)
print(f"✅ {out_path}  ({len(audio[0])/24000:.1f}초, 생성 {time.time()-t0:.1f}초)")
show_with_download(out_path)
if AUTO_DOWNLOAD:
    files.download(out_path)

In [ ]:
#@title 6. (선택) 여러 줄 대본 일괄 생성 — 줄마다 wav + 합본 wav + zip
SCRIPT = """
첫 번째 문장입니다.
두 번째 문장입니다.
세 번째 문장입니다.
"""
GAP_SEC = 0.4  #@param {type:"number"}
PREFIX = "line"  #@param {type:"string"}

import os, shutil, time
import numpy as np
import soundfile as sf
from google.colab import files
from IPython.display import Audio, HTML, display
import base64

def show_with_download(path, mime="audio/wav"):
    # 플레이어 + 다운로드 버튼 표시
    if mime.startswith("audio"):
        display(Audio(path))
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    fn = os.path.basename(path)
    display(HTML(f'<a download="{fn}" href="data:{mime};base64,{b64}" '
                 f'style="display:inline-block;padding:8px 16px;background:#1a73e8;color:#fff;'
                 f'border-radius:6px;text-decoration:none;font-weight:bold">⬇ 다운로드: {fn}</a>'))

lines = [l.strip() for l in SCRIPT.strip().splitlines() if l.strip()]
out_dir = f"/content/outputs/{PREFIX}_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)
pieces, gap = [], np.zeros(int(24000 * GAP_SEC), dtype=np.float32)
for i, line in enumerate(lines, 1):
    wav = model.generate(text=line, voice_clone_prompt=voice_prompt, speed=SPEED, num_step=NUM_STEP)[0]
    sf.write(f"{out_dir}/{PREFIX}_{i:03d}.wav", wav, 24000)
    pieces += [wav.astype(np.float32), gap]
    print(f"[{i}/{len(lines)}] {line}")
sf.write(f"{out_dir}/{PREFIX}_all.wav", np.concatenate(pieces[:-1]), 24000)
show_with_download(f"{out_dir}/{PREFIX}_all.wav")
zip_path = shutil.make_archive(out_dir, "zip", out_dir)
show_with_download(zip_path, "application/zip")
files.download(zip_path)

### 팁
- 텍스트에 `[laughter]` 같은 비언어 태그를 넣을 수 있습니다.
- 숫자를 읽게 하려면 `model.generate(..., normalize_text=True)` (영어/중국어 위주).
- 다음 세션에서 목소리 재사용: `from omnivoice import VoiceClonePrompt; voice_prompt = VoiceClonePrompt.load("/content/my_voice.pt")` (런타임이 끝나면 `/content`는 지워지니 `my_voice.pt`를 받아두거나 Drive에 복사하세요).